<a href="https://colab.research.google.com/github/julianvanegas/DB-Structure/blob/main/Arboles%20ABB%20y%20B%2B/Arboles_ABB_B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Árboles ABB & B+

In [18]:
!pip install faker

from faker import Faker   # Generador de datos falsos
import random             # Para generación de números aleatorios
import time               # Para medir tiempos de ejecución
import bisect             # Para búsqueda e inserción binaria en listas ordenadas

## Generación de datos

Generción aleatoria del listado de estudiantes, con la estructura:

```
id: identificador numérico único
name: nombre del estudiante
average: promedio de una calificación entre 0.0 y 5.0
```

In [19]:
fake = Faker("es_CO")

def generate_data(n=10000):
    """
    Genera una lista de n estudiantes con datos aleatorios.
    Cada estudiante tiene: id único, nombre y promedio académico.
    """
    # random.sample garantiza que todos los IDs generados sean únicos
    unique_ids = random.sample(range(10000, 99999), n)
    students = []

    for id in unique_ids:
        students.append({
            "id": id,
            "name": fake.name().capitalize(),
            "average": round(random.uniform(0.0, 5.0), 1)
        })
    return students


## Lista

Estructura más simple que recorre todos los elementos uno a uno hasta encontrar el buscado, es una busqueda lineal. Complejidad $O(n)$.

In [20]:
class ListSystem:
    def __init__(self, data):
        # Almacena la lista de estudiantes tal como se recibe
        self.data = data

    def search(self, student_id):
        """
        Búsqueda lineal: recorre la lista elemento por elemento.
        Peor caso: O(n) — debe revisar todos los elementos si no encuentra el ID.
        """
        for s in self.data:
            if s['id'] == student_id:
                return s  # Retorna el estudiante al encontrarlo
        return None  # Retorna None si el ID no existe en la lista


## Árbol ABB (Árbol Binario de Búsqueda)

Cada nodo tiene a lo sumo dos hijos, donde los menores van a la izquierda y los mayores a la derecha. Complejidad promedio $O(\log n)$, y el peor caso $O(n)$ si el árbol queda desbalanceado.

In [21]:
class ABBNode:
    """Representa un nodo individual dentro del árbol binario de búsqueda."""
    def __init__(self, student):
        self.id = student["id"]    # Clave de búsqueda (ID del estudiante)
        self.data = student        # Datos completos del estudiante
        self.left = None           # Hijo izquierdo (IDs menores)
        self.right = None          # Hijo derecho (IDs mayores)


class ABB:
    """Árbol Binario de Búsqueda (ABB) para gestionar estudiantes por ID."""

    def __init__(self):
        self.root = None  # El árbol comienza vacío

    def insert(self, student):
        """
        Inserta un nuevo estudiante en el árbol de forma iterativa.
        """
        new_node = ABBNode(student)

        # Caso base: árbol vacío, el nuevo nodo se convierte en raíz
        if not self.root:
            self.root = new_node
            return

        # Recorrer el árbol iterativamente hasta encontrar la posición correcta
        curr = self.root
        while True:
            if student["id"] < curr.id:
                # El ID es menor: ir al subárbol izquierdo
                if curr.left is None:
                    curr.left = new_node  # Posición libre: insertar aquí
                    return
                curr = curr.left         # Seguir bajando por la izquierda
            elif student["id"] > curr.id:
                # El ID es mayor: ir al subárbol derecho
                if curr.right is None:
                    curr.right = new_node  # Posición libre: insertar aquí
                    return
                curr = curr.right          # Seguir bajando por la derecha
            else:
                return  # ID duplicado: ignorar la inserción

    def search(self, student_id):
        """
        Búsqueda iterativa en el árbol: en cada nodo decide si bajar a la izquierda o a la derecha según el ID buscado.
        """
        curr = self.root
        while curr:
            if student_id == curr.id:
                return curr.data  # ID encontrado: retornar datos del estudiante
            # Ir a la izquierda si el ID buscado es menor, derecha si es mayor
            curr = curr.left if student_id < curr.id else curr.right
        return None  # ID no encontrado en el árbol

    def list_all(self):
        """
        Retorna todos los estudiantes ordenados por ID (recorrido inorden).
        El recorrido inorden en un ABB siempre produce los elementos en orden ascendente.
        """
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        """
        Recorrido inorden (izquierda a raíz a derecha) de forma recursiva.
        Acumula los nodos en 'result' en orden ascendente de ID.
        """
        if node:
            self._inorder(node.left, result)   # Visitar subárbol izquierdo
            result.append(node.data)           # Visitar nodo actual
            self._inorder(node.right, result)  # Visitar subárbol derecho


## Árbol B+

Variante del árbol B donde todos los datos se almacenan en las hojas, y las hojas están enlazadas entre sí. Ideal para búsquedas por rango y acceso secuencial. Complejidad $O(\log n$).

In [22]:
class BPlusNode:
    """Representa un nodo del árbol B+. Puede ser hoja o nodo interno."""
    def __init__(self, leaf=False):
        self.leaf = leaf        # True si es nodo hoja, False si es nodo interno
        self.keys = []          # Lista de claves (IDs) almacenadas en el nodo
        self.values = []        # Solo para hojas: datos completos de los estudiantes
        self.children = []      # Solo para nodos internos: referencias a nodos hijos
        self.next = None        # Enlace al siguiente nodo hoja (lista enlazada de hojas)


class BPlusTree:
    """Árbol B+ para búsqueda eficiente de estudiantes por ID."""

    def __init__(self, order=100):
        self.root = BPlusNode(leaf=True)  # El árbol inicia con una sola hoja vacía
        self.order = order  # Capacidad máxima de claves por nodo antes de dividir

    def _find_leaf_with_path(self, key):
        """
        Navega desde la raíz hasta la hoja y registra el camino en lugar de buscar el padre después.
        Retorna (hoja, pila) donde pila[-1] es el padre inmediato de la hoja.
        """
        node = self.root
        path = []  # Pila de nodos internos recorridos (ancestros)
        while not node.leaf:
            path.append(node)  # Guardar el nodo actual antes de descender
            i = bisect.bisect_right(node.keys, key)
            node = node.children[i]  # Descender al hijo correspondiente
        return node, path  # Retornar la hoja y el camino recorrido

    def _find_leaf(self, key):
        """
        Navega desde la raíz hasta la hoja sin registrar el camino.
        Solo se usa en búsquedas (search), donde el padre no es necesario.
        """
        node = self.root
        while not node.leaf:
            i = bisect.bisect_right(node.keys, key)
            node = node.children[i]
        return node

    def insert(self, student):
        """
        Inserta un estudiante en el árbol B+ usando _find_leaf_with_path para obtener el padre directamente,
        """
        key = student['id']

        # Localizar la hoja y registrar el camino de ancestros
        leaf, path = self._find_leaf_with_path(key)

        # Encontrar la posición de inserción usando búsqueda binaria
        i = bisect.bisect_left(leaf.keys, key)

        # Verificar si el ID ya existe (no se permiten duplicados)
        if i < len(leaf.keys) and leaf.keys[i] == key:
            return  # Duplicado: no insertar

        # Insertar clave y valor manteniendo el orden
        leaf.keys.insert(i, key)
        leaf.values.insert(i, student)

        # Si la hoja se desborda, dividirla pasando el camino de padres
        if len(leaf.keys) > self.order:
            self._split_leaf(leaf, path)

    def _split_leaf(self, leaf, path):
        """
        Divide una hoja llena en dos hojas recibiendo 'path' en lugar de buscar el padre.
        """
        mid = len(leaf.keys) // 2  # Punto de división

        # Crear nueva hoja con la mitad derecha
        new_leaf = BPlusNode(leaf=True)
        new_leaf.keys = leaf.keys[mid:]
        new_leaf.values = leaf.values[mid:]
        new_leaf.next = leaf.next  # Mantener el enlace de la lista de hojas

        # La hoja original retiene solo la mitad izquierda
        leaf.keys = leaf.keys[:mid]
        leaf.values = leaf.values[:mid]
        leaf.next = new_leaf  # Enlazar la hoja original a la nueva

        # Propagar la clave de separación al padre usando la pila de ancestros
        self._insert_into_parent(leaf, new_leaf.keys[0], new_leaf, path)

    def _insert_into_parent(self, left, key, right, path):
        """
        Inserta una clave en el nodo padre después de una división.
        Obtiene el padre desde 'path' en lugar de buscarlo (O(n)).
        Si la pila está vacía, el nodo dividido era la raíz y se crea una nueva.
        """
        if not path:
            # Pila vacía: el nodo dividido era la raíz, crear una nueva raíz
            new_root = BPlusNode(leaf=False)
            new_root.keys = [key]
            new_root.children = [left, right]
            self.root = new_root
            return

        # El padre es el último nodo en la pila de ancestros
        parent = path.pop()

        # Insertar la clave en el padre en la posición correcta
        i = bisect.bisect_left(parent.keys, key)
        parent.keys.insert(i, key)
        parent.children.insert(i + 1, right)  # Nuevo hijo a la derecha de la clave

        # Si el padre también se desborda, dividirlo pasando la pila restante
        if len(parent.keys) > self.order:
            self._split_internal(parent, path)

    def _split_internal(self, node, path):
        """
        Divide un nodo interno lleno en dos nodos internos.
        Recibiendo 'path' para continuar propagando hacia arriba sin buscar padres.
        La clave del medio sube al padre; no se duplica (diferencia con hojas).
        """
        mid = len(node.keys) // 2
        mid_key = node.keys[mid]  # Clave del medio que subirá al padre

        # Crear nuevo nodo interno con la mitad derecha
        new_node = BPlusNode(leaf=False)
        new_node.keys = node.keys[mid + 1:]
        new_node.children = node.children[mid + 1:]

        # El nodo original retiene solo la mitad izquierda
        node.keys = node.keys[:mid]
        node.children = node.children[:mid + 1]

        # La clave del medio sube al padre, pasando la pila de ancestros restante
        self._insert_into_parent(node, mid_key, new_node, path)

    def search(self, student_id):
        """
        Busca un estudiante por ID en el árbol B+.
        Navega hasta la hoja correspondiente y verifica si el ID está allí.
        """
        leaf = self._find_leaf(student_id)  # Encontrar la hoja que debería contener el ID
        i = bisect.bisect_left(leaf.keys, student_id)  # Posición del ID en la hoja

        # Verificar que el ID encontrado en esa posición coincide exactamente
        if i < len(leaf.keys) and leaf.keys[i] == student_id:
            return leaf.values[i]  # Retornar los datos del estudiante
        return None  # ID no encontrado


## Comparación entre los árboles

Se comparan los tiempos de ejecución para búsquedas de cada algoritmo de los árboles ABB y B+.

In [23]:
def benchmark(qty, structures):
    """
    Mide y compara el tiempo de búsqueda de cada estructura de datos.
    Realiza dos pruebas: búsquedas en orden aleatorio y en orden ascendente.
    """
    # Extraer todos los IDs disponibles y seleccionar una muestra aleatoria
    all_ids = [s['id'] for s in students]
    random_ids = random.sample(all_ids, qty)   # IDs en orden aleatorio
    ordered_ids = sorted(random_ids)           # Los mismos IDs pero en orden ascendente

    def measure_time(system, ids_list):
        """
        Mide el tiempo total de búsqueda de una lista de IDs en una estructura.
        Retorna el tiempo en milisegundos.
        """
        start = time.perf_counter()    # Inicio del cronómetro (alta precisión)
        for sid in ids_list:
            system.search(sid)         # Realizar cada búsqueda
        end = time.perf_counter()      # Fin del cronómetro
        return (end - start) * 1000    # Convertir de segundos a milisegundos

    # Imprimir encabezado de la tabla de resultados
    print(f"--- Comparación con {qty} búsquedas ---")
    print(f"{'Estructura':<10} | {'Búsqueda Aleatoria (ms)':<23} | {'Búsqueda Ordenada (ms)':<22}")
    print("-" * 62)

    # Medir y mostrar el tiempo de cada estructura
    for name, system in structures:
        t_rand = measure_time(system, random_ids)   # Tiempo con IDs aleatorios
        t_ord = measure_time(system, ordered_ids)   # Tiempo con IDs ordenados
        print(f"{name:<10} | {t_rand:<23.4f} | {t_ord:<22.4f}")

    print("\n")


## Ejecución y resultados

In [25]:
if __name__ == '__main__':

    # 1. Generar 10.000 estudiantes con datos aleatorios
    students = generate_data(10000)

    # 2. Inicializar las tres estructuras de datos
    list_system = ListSystem(students)  # Lista simple (ya cargada en el constructor)
    abb = ABB()                         # Árbol Binario de Búsqueda (vacío)
    bplus = BPlusTree()                 # Árbol B+ (vacío)

    # 3. Insertar todos los estudiantes en los árboles
    for student in students:
        abb.insert(student)    # Inserción en el ABB
        bplus.insert(student)  # Inserción en el B+

    # 4. Definir las estructuras a comparar (nombre + objeto)
    structures = [
        ("Lista",    list_system),
        ("ABB",      abb),
        ("B+",       bplus)
    ]

    # 5. Ejecutar el benchmark con distintas cantidades de búsquedas
    benchmark(100,  structures)   # Prueba con 100 búsquedas
    benchmark(1000, structures)   # Prueba con 1.000 búsquedas
    benchmark(5000, structures)   # Prueba con 5.000 búsquedas


--- Comparación con 100 búsquedas ---
Estructura | Búsqueda Aleatoria (ms) | Búsqueda Ordenada (ms)
--------------------------------------------------------------
Lista      | 22.5101                 | 22.7412               
ABB        | 0.2108                  | 0.1054                
B+         | 0.1560                  | 0.0621                


--- Comparación con 1000 búsquedas ---
Estructura | Búsqueda Aleatoria (ms) | Búsqueda Ordenada (ms)
--------------------------------------------------------------
Lista      | 233.5339                | 239.0119              
ABB        | 1.5584                  | 1.1227                
B+         | 0.8890                  | 0.5860                


--- Comparación con 5000 búsquedas ---
Estructura | Búsqueda Aleatoria (ms) | Búsqueda Ordenada (ms)
--------------------------------------------------------------
Lista      | 1813.3489               | 1860.2129             
ABB        | 7.3351                  | 6.0067                
B+       